# Getting started with VAFT.jl

VAFT.jl is the Julia-native interface to the Versatile Analysis Framework for Tokamak.
It reads IMAS-compatible files into the native data structures of
[IMASdd.jl](https://github.com/ProjectTorreyPines/IMASdd.jl), so everything you load can be used
directly with IMASdd.jl. No Python is needed.

This notebook covers:

1. checking the environment,
2. loading a file,
3. exploring the data, including units and uncertainties,
4. time-based access with IMASdd.jl,
5. writing and reading back,
6. what errors look like.

Open it from inside the VAFT.jl repository with a Julia kernel (see `notebooks/README.md`).

## 1. Setup

`VAFT.doctor()` reports versions and supported formats. It only reads the environment.

In [1]:
using VAFT
const IMASdd = VAFT.IMASdd

VAFT.doctor();

VAFT.jl environment


  Julia     : 1.12.5
  VAFT.jl   : 0.1.0
  IMASdd.jl : 8.6.2
  HDF5.jl   : 0.17.4
  Formats   :
    :json OMAS/IMAS JSON (.json) via IMASdd.json2imas
    :h5i  IMAS tensorized HDF5 (.h5, .hdf5) via IMASdd.h5i2imas
    :hdf  OMAS hierarchical HDF5 (.h5, .hdf5) via IMASdd.hdf2imas
  Python    : PythonCall.jl not installed (optional; not required)


## 2. Load a file

`load_imas` takes a local path and returns an `IMASdd.dd`. The format is inferred from the extension
(`.json`, `.h5`, `.hdf5`); for HDF5 it also tells IMAS tensorized from OMAS hierarchical layout.
Use `format=:json`, `:h5i` or `:hdf` to override.

We use the small synthetic equilibrium shipped with the tests.

In [2]:
path = joinpath(pkgdir(VAFT), "test", "fixtures", "equilibrium_small.json")
dd = load_imas(path)
typeof(dd)

IMASdd.dd{Float64}

## 3. Explore the data

The result is an ordinary IMASdd.jl structure: use field access and indexing.

In [3]:
eq = dd.equilibrium

println("comment     : ", eq.ids_properties.comment)
println("time        : ", eq.time, " ", IMASdd.units(eq, :time))
println("time slices : ", length(eq.time_slice))

comment     : VAFT.jl test fixture
time        : 

[0.1, 0.2] s
time slices : 2


Units come from the IMAS Data Dictionary. Values are returned as stored; VAFT.jl does not convert them.
Uncertainties are accessed with the `_σ` suffix.

In [4]:
using Printf

@printf("%-8s %-14s %-12s\n", "t [s]", "ip [A]", "σ(ip) [A]")
for ts in eq.time_slice
    gq = ts.global_quantities
    @printf("%-8.2f %-14.1f %-12.1f\n", ts.time, gq.ip, gq.ip_σ)
end

t [s]    ip [A]         σ(ip) [A]   
0.10     -100000.0      1000.0      


0.20     -200000.0      2000.0      


In [5]:
p1 = eq.time_slice[1].profiles_1d
println("psi      [", IMASdd.units(p1, :psi), "] = ", p1.psi)
println("pressure [", IMASdd.units(p1, :pressure), "] = ", p1.pressure)

psi      [Wb] = [0.0, 0.25, 0.5, 0.75, 1.0]
pressure [Pa] = [1000.0, 750.0, 500.0, 250.0, 0.0]


## 4. Time-based access

IMASdd.jl keeps a global time on each `dd`. Indexing a time-dependent array with `[]` returns the
element at that time, and `[t]` at an explicit time. Both pick the most recent slice at or before
the requested time (causal lookup).

In [6]:
IMASdd.global_time(dd, 0.15)
println("global time  = ", IMASdd.global_time(dd))
println("slice at []  : t = ", eq.time_slice[].time)
println("slice at 0.2 : t = ", eq.time_slice[0.2].time)

global time  = 0.15
slice at []  : t = 0.1
slice at 0.2 : t = 0.2


## 5. Write and read back

Writing is done with IMASdd.jl directly; VAFT.jl adds no serializers. Both HDF5 files below use
the `.h5` extension, and `load_imas` works out on its own which layout each one has.

In [7]:
dir = mktempdir()
writers = [
    "tensorized.h5"   => IMASdd.imas2h5i,
    "hierarchical.h5" => IMASdd.imas2hdf,
    "copy.json"       => IMASdd.imas2json,
]
for (name, writer) in writers
    file = joinpath(dir, name)
    writer(dd, file)
    dd2 = load_imas(file)
    same = dd2.equilibrium.time_slice[2].global_quantities.ip == eq.time_slice[2].global_quantities.ip &&
           dd2.equilibrium.time_slice[2].profiles_1d.pressure == eq.time_slice[2].profiles_1d.pressure
    println(rpad(name, 16), " values match: ", same)
end

tensorized.h5    values match: 

true
hierarchical.h5  values match: 

true
copy.json        values match: 

true


## 6. Errors

Unsupported formats and read failures raise Julia exceptions with an explanation:
`VAFT.UnsupportedFormatError` and `VAFT.IMASLoadError` (the latter keeps the original error in `.cause`).

In [8]:
cd(dir) do
    write("notes.txt", "not IMAS data")
    try
        load_imas("notes.txt")
    catch err
        showerror(stdout, err)
    end
end

UnsupportedFormatError: cannot infer an IMAS file format from the extension of "notes.txt". Supported formats: `:json` (OMAS/IMAS JSON; .json), `:h5i` (IMAS tensorized HDF5; .h5, .hdf5), `:hdf` (OMAS hierarchical HDF5; .h5, .hdf5). Pass `format=` explicitly for files with other extensions.

## Next steps

- `?load_imas` and `?VAFT.doctor` for the full API.
- [IMASdd.jl documentation](https://projecttorreypines.github.io/IMASdd.jl/dev/) for everything you can do with `dd`.
- [`docs/architecture.md`](../docs/architecture.md): how VAFT.jl relates to IMASdd.jl and VAFT's DD/DDView design.
- [`docs/interoperability.md`](../docs/interoperability.md): plans for Python VAFT interoperability.
- [README](../README.md#known-limitations): known limitations.